In [1]:
pip install plotly

                                              0.0/9.9 MB ? eta -:--:--
     -                                        0.4/9.9 MB 12.9 MB/s eta 0:00:01
     ----                                     1.0/9.9 MB 12.6 MB/s eta 0:00:01
     -----                                    1.4/9.9 MB 11.1 MB/s eta 0:00:01
     -------                                  2.0/9.9 MB 12.6 MB/s eta 0:00:01
     ---------                                2.4/9.9 MB 11.9 MB/s eta 0:00:01
     -----------                              3.0/9.9 MB 11.8 MB/s eta 0:00:01
     --------------                           3.6/9.9 MB 12.0 MB/s eta 0:00:01
     ----------------                         4.2/9.9 MB 12.1 MB/s eta 0:00:01
     -------------------                      4.7/9.9 MB 11.6 MB/s eta 0:00:01
     ---------------------                    5.3/9.9 MB 11.7 MB/s eta 0:00:01
     -----------------------                  5.8/9.9 MB 11.8 MB/s eta 0:00:01
     -------------------------                6.3/9.9 MB 11.


[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# caminho + supressão de avisos
import os, warnings

# manipulação de dados
import pandas as pd
import numpy as np 

# conexão com o banco 
from sqlalchemy import create_engine

# aprendizado de maquina
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# visualização - dashboard
import plotly.graph_objects as go 
import plotly.subplots as make_subplots

In [3]:
warnings.filterwarnings('ignore')

In [5]:
# conectar o banco 
ENGINE_URL = ('mysql+pymysql://root:@localhost:3306/bolsa_familia')

engine = create_engine(ENGINE_URL, echo=False)

query = ''' 
        SELECT `MÊS COMPETÊNCIA`, `UF`, 
        COUNT(*) AS qtd_parcelas,
        AVG(`VALOR PARCELA`) AS valor_medio,
        SUM(`VALOR PARCELA`) AS valor_total  
        FROM bolsa_familia
        GROUP BY `MÊS COMPETÊNCIA`, `UF`
        '''
df = pd.read_sql(query, engine)

In [7]:
# EDA 
df_uf = df.groupby('UF').agg(
    media_valor = ('valor_medio', 'mean'), 
    total_parcelas = ('qtd_parcelas', 'sum')
    ).reset_index()


In [ ]:
# calcular 
serie = df_uf['media_valor']
q1, q2, q3 = np.percentile(serie, [25, 50, 75])
iqr = q3 - q1
print(f'Media: {serie.mean():.2f} // Mediana: {serie.median():.2f}')
print(f'Q1: {q1}, Q3: {q3}, IQR: {iqr}')
print(f'Assimetria: {serie.skew():.3f}')
print(f'Curtose: {serie.kurt():.3f}')


Media: 675.99 // Mediana: 665.70
Q1: 659.6646461509814, Q3: 681.7392755192981, IQR: 22.074629368316664
Assimetria: 1.362
Curtose: 0.870


In [ ]:
# aprendizado // agrupamento // não supervisionado 
features = df_uf[['media_valor','qtd_parcelas']].values

# boas práticas para aprendizado de maquina 
scaler = StandardScaler()
features_norm = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df_uf['cluster'] = kmeans.fit_predict(features_norm)
print(df_uf.groupby('cluster')[['UF', 'media_valor']])

KeyError: "['qtd_parcelas'] not in index"